In [2]:
import tomllib
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, URL
from dotenv import dotenv_values

OUTPUT_FOLDER = Path("input")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
dotenv = dotenv_values()

with open("./config.toml", "rb") as f:
    config = tomllib.load(f)

def connect_sql(database=None):
    """Establish a reusable SQLAlchemy engine."""
    url = URL.create("mysql+pymysql", username=dotenv['SQL_USER'],
                     password=dotenv['SQL_PASSWORD'], host=config['sql']['host'],
                     port=config['sql']['port'], database=database)
    return create_engine(url)

engine = connect_sql(database=config["sql"]["schema"])

In [ ]:
# Next the thing with the Chinese patents sharing the same inpadoc but different docdb families
# And then the patent novelty indicator

# whether Chinese patents belonging to different DOCDB families within the same INPADOC family represent
# new technical contributions or variations of existing inventions.

In [ ]:
# get pubnrs

In [3]:
query1 = """
WITH base AS (
SELECT
  t201.appln_id,
  t201.INPADOC_FAMILY_ID,
  t201.DOCDB_FAMILY_ID
FROM TLS201_APPLN t201
WHERE t201.APPLN_AUTH = 'CN'
AND t201.INPADOC_FAMILY_ID IN (
    SELECT 
      INPADOC_FAMILY_ID
    FROM TLS201_APPLN
    WHERE APPLN_AUTH = 'CN'
    GROUP BY INPADOC_FAMILY_ID
    HAVING COUNT(DISTINCT DOCDB_FAMILY_ID) > 1
))

SELECT
  base.INPADOC_FAMILY_ID AS inpadoc_family_id,
  base.DOCDB_FAMILY_ID AS docdb_family_id,
  t211.PUBLN_AUTH AS publn_auth,
  t211.PUBLN_NR AS publn_nr,
  t211.PUBLN_KIND AS publn_kind
FROM base
LEFT JOIN TLS211_PAT_PUBLN t211 ON base.appln_id = t211.appln_id
ORDER BY
    base.INPADOC_FAMILY_ID,
    base.DOCDB_FAMILY_ID
"""

In [ ]:
with engine.connect() as connection:
    out1 = pd.read_sql(query1, connection)

In [ ]:
out1.to_parquet(OUTPUT_FOLDER / 'CN_pubnrs.parquet', engine = 'pyarrow', index = False)

In [ ]:
# get abstracts

In [3]:
query2 = """
WITH base AS (
SELECT
  t201.APPLN_ID
FROM TLS201_APPLN t201
WHERE t201.APPLN_AUTH = 'CN'
AND t201.INPADOC_FAMILY_ID IN (
    SELECT 
      INPADOC_FAMILY_ID
    FROM TLS201_APPLN
    WHERE APPLN_AUTH = 'CN'
    GROUP BY INPADOC_FAMILY_ID
    HAVING COUNT(DISTINCT DOCDB_FAMILY_ID) > 1
))

SELECT
  base.APPLN_ID as appln_id,
  t203.APPLN_ABSTRACT_LG as abstract_lg,
  t203.APPLN_ABSTRACT as abstract
FROM base
LEFT JOIN TLS203_APPLN_ABSTR t203 ON base.APPLN_ID = t203.APPLN_ID
"""

In [4]:
with engine.connect() as connection:
    out2 = pd.read_sql(query2, connection)

In [6]:
out2.to_parquet(OUTPUT_FOLDER / 'CN_abstracts.parquet', engine = 'pyarrow', index = False)

---
---

In [ ]:
# Sanity

In [2]:
out = pd.read_csv('_3_chinese_patents/CN_patent_novelty.csv')

In [3]:
out.sample(5)

,inpadoc_family_id,docdb_family_id,publn_auth,publn_nr,publn_kind
471992,487092501,60486499,CN,107451178.0,B
825784,585184202,86324569,CN,118265974.0,A
284194,407641972,53346011,CN,104698730.0,A
954413,623632925,93991772,CN,222275946.0,U
149124,50574342,51262811,CN,104969368.0,A


In [4]:
out.shape

(1007544, 5)

In [14]:
with engine.connect() as connection:
    s1 = """
    SELECT
        appln_id,
        INPADOC_FAMILY_ID,
        DOCDB_FAMILY_ID
    FROM TLS201_APPLN
    WHERE INPADOC_FAMILY_ID = 585184202 
    AND APPLN_AUTH = 'CN'
    """
    s1 = pd.read_sql(s1, connection)

In [15]:
appln_ids = ','.join(s1['appln_id'].astype('str'))

In [16]:
appln_ids

'613915012,615603878'

In [17]:
with engine.connect() as connection:
    s2 = f"""
    SELECT
        appln_id,
        PUBLN_AUTH,
        PUBLN_NR,
        PUBLN_KIND
    FROM TLS211_PAT_PUBLN
    WHERE appln_id IN ({appln_ids}) 
    """
    s2 = pd.read_sql(s2, connection)

In [ ]:
s3 = out.groupby('inpadoc_family_id').agg(nr_docdb=('docdb_family_id', 'nunique')).reset_index()

In [ ]:
s3['nr_docdb'].min()

np.int64(2)